In [48]:
import numpy as np
import pandas as pd

In [49]:
df = pd.read_csv("DateFruit_Dataset.csv")
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [50]:
df.isnull().sum()

AREA             0
PERIMETER        0
MAJOR_AXIS       0
MINOR_AXIS       0
ECCENTRICITY     0
EQDIASQ          0
SOLIDITY         0
CONVEX_AREA      0
EXTENT           0
ASPECT_RATIO     0
ROUNDNESS        0
COMPACTNESS      0
SHAPEFACTOR_1    0
SHAPEFACTOR_2    0
SHAPEFACTOR_3    0
SHAPEFACTOR_4    0
MeanRR           0
MeanRG           0
MeanRB           0
StdDevRR         0
StdDevRG         0
StdDevRB         0
SkewRR           0
SkewRG           0
SkewRB           0
KurtosisRR       0
KurtosisRG       0
KurtosisRB       0
EntropyRR        0
EntropyRG        0
EntropyRB        0
ALLdaub4RR       0
ALLdaub4RG       0
ALLdaub4RB       0
Class            0
dtype: int64

In [51]:
df.shape 

(898, 35)

In [52]:
df["Class"].unique()  # 7 unique classes

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [53]:
X = df.drop("Class",axis=1)
y = df["Class"]

In [54]:
# encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_encoded = le.fit_transform(y)

In [55]:
# train test split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y_encoded,test_size=0.2,random_state=42
)


In [56]:
# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [57]:
type(y_train)

numpy.ndarray

### ANN

In [58]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [59]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)

y_train_tensor = torch.tensor(y_train,dtype = torch.long)
y_test_tensor = torch.tensor(y_test,dtype = torch.long)

''' 
in multiclass classification we use cross entropy loss as a 
loss function that expect target value as a long datatype so 
we set target(y) as long
  '''

' \nin multiclass classification we use cross entropy loss as a \nloss function that expect target value as a long datatype so \nwe set target(y) as long\n  '

In [60]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor) 


In [61]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

In [62]:
# Build our model 

class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer 
            nn.Linear(X.shape[1],64),
            nn.ReLU(),


            # 2nd hidden layer
            nn.Linear(64,64),
            nn.ReLU(),


            # output layer
            nn.Linear(64,7)

        
            # here we are going to use crossEntropyLoss 
            # for loss function and this loss function 
            # automatically applies softmax so whenever 
            # we are using loss function CrossEntropyLoss 
            # we do not have to specifically mention 
            # Softmax in output layer while defining ANN model
            
            
            # when we will use softmax function explicitely in case of Multiclass Classification
            # 1) We want to calculate Probabilities
            # 2) We are not using CrossEntropyLoss as a loss function
            
        
        )

    def forward(self,x):
        return self.model(x)

In [63]:
model = ANN()

# loss & optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [64]:
# Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()
    
    running_loss=0.0
    best_model_loss = float("inf")

    for xb,yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs,yb)
        loss.backward()
        optimizer.step() # params update

        running_loss+=loss.item()

    train_loss = running_loss/len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

    model.eval()

    running_eval_loss = 0.0
    
    with torch.no_grad():
        for xb,yb in test_loader:
            outputs = model(xb)
            loss = criteria(outputs,yb)
            running_eval_loss += loss.item()
    
    eval_loss = running_eval_loss/len(test_loader)
    if eval_loss < best_model_loss :
        best_model_loss = eval_loss
        torch.save(model.state_dict(),"best_model.pt")

epoch = 1/100, loss = 1.7253400916638582
epoch = 2/100, loss = 1.0836288618004841
epoch = 3/100, loss = 0.7292203501514767
epoch = 4/100, loss = 0.5503649905971859
epoch = 5/100, loss = 0.4414018593404604
epoch = 6/100, loss = 0.37740758320559625
epoch = 7/100, loss = 0.33213724876227585
epoch = 8/100, loss = 0.30651200206383417
epoch = 9/100, loss = 0.27342018031555676
epoch = 10/100, loss = 0.27241178440011066
epoch = 11/100, loss = 0.24487361214731052
epoch = 12/100, loss = 0.22521676544262015
epoch = 13/100, loss = 0.2106063226642816
epoch = 14/100, loss = 0.19574914321951245
epoch = 15/100, loss = 0.19210182843000992
epoch = 16/100, loss = 0.18082042953566366
epoch = 17/100, loss = 0.17702801784743433
epoch = 18/100, loss = 0.1727842393776645
epoch = 19/100, loss = 0.16538026281025098
epoch = 20/100, loss = 0.15688700147944948
epoch = 21/100, loss = 0.15159795935387196
epoch = 22/100, loss = 0.14933178185120874
epoch = 23/100, loss = 0.14608029324723326
epoch = 24/100, loss = 0.14

In [65]:
# load the best model

model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [71]:
# Evaluate

model.eval()

total=0
correct=0

with torch.no_grad():
    for xb,yb in test_loader:
        outputs = model(xb) # [0.2,0.5,..] -7 vals
        _,predicted = torch.max(outputs,1)
        
        correct += (predicted==yb).sum().item()
        total += yb.size(0) # actual samples in each batch

print("accuracy_score = ",correct/total *100)

accuracy_score =  93.33333333333333


In [ ]:
# alternate way
from sklearn.metrics import accuracy_score
model.eval()
outputs = model(X_test_tensor)
_,predicted = torch.max(outputs,1)
print("accuracy score = ",accuracy_score(y_test,predicted))

accuracy score =  0.9333333333333333


In [ ]:
# alternate way
correct=0
total=0
model.eval()
outputs = model(X_test_tensor)
_,predicted = torch.max(outputs,1)
total = len(y_test)
correct = (predicted == y_test).sum().item()

print(" accuracy score ",correct/total)

 accuracy score  0.9333333333333333


### PCA

In [ ]:
from sklearn.